## Phase 0 - Imports & Configuration

In [ ]:
import os
import json
import re
import string as _string
import time
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import stanza
import torch
from captum.attr import IntegratedGradients
from sklearn.metrics import (
    accuracy_score, average_precision_score,
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, f1_score,
    precision_recall_curve, precision_score,
    recall_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainerCallback,
    TrainingArguments,
    Trainer,
)

In [ ]:
DATASET_PATH = "../PRDECT-ID Dataset.csv"
_SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
MODEL_DIR   = os.path.join(_SCRIPT_DIR, "model_weights")
EVAL_DIR    = os.path.join(_SCRIPT_DIR, "evaluation_metrics")
RESULTS_DIR = os.path.join(_SCRIPT_DIR, "results")
INDOBERT_MODEL_PATH = os.path.join(MODEL_DIR, "indobert_model")

for d in (MODEL_DIR, EVAL_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok=True)

MODEL_NAME = "indobenchmark/indobert-base-p1"
LABEL2ID   = {"negatif": 0, "positif": 1}
ID2LABEL   = {0: "negatif", 1: "positif"}

CONFIDENCE_THRESHOLD = 0.75
SIMILARITY_THRESHOLD = 0.99
TOP_N_SUMMARY        = 10
MIN_COUNT            = 3

NEGATION_WORDS = {"tidak", "bukan", "belum", "jangan", "kurang", "tanpa"}
SPLIT_CONJUNCTIONS = ["tapi", "tetapi", "namun", "meski", "meskipun", "walaupun", "walau", "dan", "serta", "juga", "karena", "soalnya", "sebab", "padahal", "sedangkan"]

REGEX_PATTERNS = [
    (r"(.)\1{2,}",          r"\1\1"),   # "bangeeetttt" -> "bangeett"
    (r"([a-zA-Z])\1\b",     r"\1"),     # "bangett" -> "banget"
    (r"([!?,;.]){2,}",      r"\1"),     # ",,,,," -> ","
    (r"\b(\w+)\s+\1\b",     r"\1"),     # "bagus bagus" -> "bagus"
    (r"\s+([!?,;.])",       r"\1"),     # space before punctuation
    (r"([!?,;.])(?!\s)",    r"\1 "),    # space after punctuation
]

SLANG_DICT = {
    "gak": "tidak", "ga": "tidak", "gk": "tidak", "nggak": "tidak", "ngga": "tidak", "engga": "tidak", "tdk": "tidak", "tak": "tidak",
    "tp": "tapi", "tpi": "tapi", "cuma": "hanya", "cmn": "hanya", "cuman": "hanya", "sgt": "sangat", "sngat": "sangat", "sangt": "sangat",
    "bgt": "banget", "bgd": "banget", "bngt": "banget", "bet": "banget", "bnget": "banget", "jg": "juga", "sm": "sama", "mk": "maka",
    "sy": "saya", "sya": "saya", "gw": "saya", "gue": "saya", "aku": "saya", "ak": "saya", "lo": "kamu", "lu": "kamu", "kmu": "kamu",
    "ud": "sudah", "udh": "sudah", "uda": "sudah", "udah": "sudah", "sdh": "sudah", "blm": "belum", "blum": "belum", "belom": "belum", "blom": "belum",
    "lg": "lagi", "lgi": "lagi", "bs": "bisa", "bsa": "bisa", "skrg": "sekarang", "skr": "sekarang", "skg": "sekarang", "cb": "coba", "cba": "coba",
    "br": "baru", "bru": "baru", "cpt": "cepat", "cepet": "cepat", "cpet": "cepat", "lm": "lama", "lma": "lama", "hrs": "harus", "hrus": "harus",
    "bgs": "bagus", "bgus": "bagus", "mntp": "bagus", "mntap": "bagus", "mantep": "bagus", "mntep": "bagus", "mantap": "bagus",
    "sj": "saja", "aja": "saja", "sja": "saja", "aj": "saja", "doank": "saja", "doang": "saja", "utk": "untuk", "buat": "untuk",
    "yg": "yang", "yng": "yang", "krn": "karena", "karna": "karena", "dgn": "dengan", "dg": "dengan", "dr": "dari", "dri": "dari",
    "brg": "barang", "brng": "barang", "pnjual": "penjual", "pngiriman": "pengiriman", "pngirim": "pengirim", "krm": "kirim",
    "ongkir": "ongkos kirim", "ori": "original", "ok": "oke", "lmyn": "lumayan", "lmayan": "lumayan", "kualits": "kualitas",
    "wkwk": "", "wkwkwk": "", "haha": "", "hihi": "", "hehe": "", "masyaallah": "", "sih": "", "mantul": "bagus banget", "gokil": "luar biasa", "okesip": "oke siap"
}

STOPWORDS = {
    "yang", "di", "dan", "ini", "itu", "dengan", "untuk", "pada", "adalah", "dari", "dalam", "ke", "akan", "oleh", "saya", "aku", "baru",
    "kamu", "dia", "kami", "mereka", "nya", "ada", "sudah", "belum", "juga", "bisa", "hanya", "lebih", "lagi", "sangat", "sekali", "saat",
    "kalau", "jika", "atau", "karena", "tapi", "tetapi", "namun", "se", "si", "pun", "lah", "kah", "dong", "deh", "sih", "nih", "waktu",
    "ya", "yah", "kok", "kan", "mau", "mah", "banget", "aja", "udah", "jadi", "sama", "satu", "dua", "tiga", "masih", "harus", "banyak",
}

IG_LEXICON: set = set()


In [ ]:
stanza.download("id", verbose=False)
nlp_stanza = stanza.Pipeline(
    "id",
    processors="tokenize,mwt,pos,lemma,depparse",
    verbose=False,
)

## Phase 1 - Data Loading & Preprocessing

In [ ]:
_CLEAN_WORD_RE = re.compile(r'^[a-zA-Z]{3,}$')

def is_clean_word(word: str) -> bool:
    return bool(_CLEAN_WORD_RE.match(word))

def apply_regex_patterns(text: str) -> str:
    for pattern, replacement in REGEX_PATTERNS:
        text = re.sub(pattern, replacement, text)
    return text

def normalize_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)

    words = text.split()
    normalized = []
    for word in words:
        replacement = SLANG_DICT.get(word, word)
        if replacement:
            normalized.extend(replacement.split())

    text = " ".join(normalized)
    text = apply_regex_patterns(text)
    text = re.sub(r"[^\w\s,.!?]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def load_and_clean_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = df.rename(columns={
        "Customer Review": "review_text",
        "Customer Rating": "rating",
    })
    df["review_text"] = df["review_text"].astype(str)
    df["rating"]      = pd.to_numeric(df["rating"], errors="coerce")
    df = df.dropna(subset=["rating"])
    df["rating"] = df["rating"].astype(int)
    df = df[df["review_text"].str.split().str.len() >= 5]
    df = df[df["review_text"].str.contains(r"[a-zA-Z]", regex=True)]
    df = df[df["review_text"].str.contains(r"[a-zA-Z0-9\s.,!?]", regex=True)]
    df = df.drop_duplicates(subset=["review_text"])
    df = df.reset_index(drop=True)
    df["review_normalized"] = df["review_text"].apply(normalize_text)

    def assign_label(sentiment: str):
        if sentiment == "Negative": return 0
        elif sentiment == "Positive": return 1
        return None

    df["label"] = df["Sentiment"].apply(assign_label)
    print(f"Total after cleaning : {len(df)} rows")
    print(f"Label distribution:")
    print(f"  Positive (1): {(df['label'] == 1).sum()}")
    print(f"  Negative (0): {(df['label'] == 0).sum()}")
    return df

df_raw = load_and_clean_data(DATASET_PATH)
df_train_pool = df_raw[df_raw["label"].notna()].copy()
df_train_pool["label"] = df_train_pool["label"].astype(int)

print()
print("Normalization samples:")
for i in range(min(3, len(df_raw))):
    print(f"ORIGINAL   : {df_raw['review_text'].iloc[i]}")
    print(f"NORMALIZED : {df_raw['review_normalized'].iloc[i]}")
    print()
    

## Phase 2 - IndoBERT Model Training

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=max_length,
        )
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

class EpochMetricsCallback(TrainerCallback):
    def __init__(self):
        self.train_loss  = []
        self.train_acc   = []
        self.train_prec  = []
        self.train_rec   = []
        self.train_f1    = []
        self.val_loss    = []
        self.val_acc     = []
        self.val_prec    = []
        self.val_rec     = []
        self.val_f1      = []
        self.epochs      = []
        self._step_losses = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        if "loss" in logs and "eval_loss" not in logs:
            self._step_losses.append(logs["loss"])
        if "eval_loss" in logs:
            epoch = logs.get("epoch", len(self.epochs) + 1)
            self.epochs.append(epoch)
            if self._step_losses:
                self.train_loss.append(float(np.mean(self._step_losses)))
                self._step_losses = []
            else:
                self.train_loss.append(float("nan"))

            self.val_loss.append(logs.get("eval_loss",     float("nan")))
            self.val_acc.append(logs.get("eval_accuracy",  float("nan")))
            self.val_prec.append(logs.get("eval_precision", float("nan")))
            self.val_rec.append(logs.get("eval_recall",    float("nan")))
            self.val_f1.append(logs.get("eval_f1",         float("nan")))
            self.train_acc.append(logs.get("train_accuracy",  float("nan")))
            self.train_prec.append(logs.get("train_precision", float("nan")))
            self.train_rec.append(logs.get("train_recall",    float("nan")))
            self.train_f1.append(logs.get("train_f1",         float("nan")))

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy" : accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, average="weighted", zero_division=0),
        "recall"   : recall_score(labels, preds, average="weighted", zero_division=0),
        "f1"       : f1_score(labels, preds, average="weighted", zero_division=0),
    }

texts  = df_train_pool["review_normalized"].tolist()
labels = df_train_pool["label"].tolist()

X_train, X_val, y_train, y_val = train_test_split(texts, labels, test_size=0.1, random_state=42, stratify=labels)
print(f"Train: {len(X_train)} | Val: {len(X_val)}")

metrics_callback = EpochMetricsCallback()

if os.path.exists(INDOBERT_MODEL_PATH) and os.path.exists(os.path.join(INDOBERT_MODEL_PATH, "config.json")):
    print("Loading saved IndoBERT model - skipping training.")
    tokenizer     = AutoTokenizer.from_pretrained(INDOBERT_MODEL_PATH)
    model         = AutoModelForSequenceClassification.from_pretrained(INDOBERT_MODEL_PATH)
    train_dataset = ReviewDataset(X_train, y_train, tokenizer)
    val_dataset   = ReviewDataset(X_val,   y_val,   tokenizer)
else:
    print("Loading base IndoBERT...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model     = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )
    train_dataset = ReviewDataset(X_train, y_train, tokenizer)
    val_dataset   = ReviewDataset(X_val,   y_val,   tokenizer)

    training_args = TrainingArguments(
        output_dir                  = "./checkpoints",
        num_train_epochs            = 3,
        per_device_train_batch_size = 32,
        per_device_eval_batch_size  = 32,
        warmup_steps                = 100,
        weight_decay                = 0.01,
        logging_dir                 = "./logs",
        logging_steps               = 50,
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        load_best_model_at_end      = True,
        metric_for_best_model       = "f1",
        fp16                        = True,
    )
    trainer = Trainer(
        model           = model,
        args            = training_args,
        train_dataset   = train_dataset,
        eval_dataset    = val_dataset,
        compute_metrics = compute_metrics,
        callbacks       = [metrics_callback],
    )
    trainer.train()
    model.save_pretrained(INDOBERT_MODEL_PATH)
    tokenizer.save_pretrained(INDOBERT_MODEL_PATH)
    print(f"IndoBERT model saved to {INDOBERT_MODEL_PATH}")

In [ ]:
def _save_epoch_plot(train_vals, val_vals, ylabel, title, filename):
    epochs_range = range(1, len(train_vals) + 1) if train_vals else []
    fig, ax = plt.subplots(figsize=(8, 5))
    if train_vals:
        if not all(np.isnan(v) for v in train_vals):
            ax.plot(epochs_range, train_vals, marker="o", color="steelblue",  label="Train")
        ax.plot(epochs_range, val_vals,   marker="s", color="darkorange", label="Validation")
    else:
        ax.text(0.5, 0.5, "No epoch data\n(model loaded from checkpoint)", ha="center", va="center", transform=ax.transAxes, color="gray")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.grid(linestyle="--", alpha=0.5)
    plt.tight_layout()
    path = os.path.join(EVAL_DIR, filename)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {path}")

if metrics_callback.val_loss:
    _save_epoch_plot(metrics_callback.train_loss, metrics_callback.val_loss, "Loss", "IndoBERT Loss - Train vs Validation", "indobert_loss.png")
    _save_epoch_plot(metrics_callback.train_acc, metrics_callback.val_acc, "Accuracy", "IndoBERT Accuracy - Train vs Validation", "indobert_accuracy.png")
    _save_epoch_plot(metrics_callback.train_prec, metrics_callback.val_prec, "Precision", "IndoBERT Precision - Train vs Validation", "indobert_precision.png")
    _save_epoch_plot(metrics_callback.train_rec, metrics_callback.val_rec, "Recall", "IndoBERT Recall - Train vs Validation", "indobert_recall.png")
    _save_epoch_plot(metrics_callback.train_f1, metrics_callback.val_f1, "F1", "IndoBERT F1 - Train vs Validation", "indobert_f1.png")
else:
    print("Training history not available (model loaded from disk) - skipping epoch plots.")

def evaluate_model(model, val_dataset, device, eval_dir: str, callback: EpochMetricsCallback) -> dict:
    model.eval()
    model.to(device)
    loader     = DataLoader(val_dataset, batch_size=32)
    all_preds  = []
    all_labels = []
    all_probs  = []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels_batch   = batch["labels"].numpy()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs   = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
            preds   = np.argmax(probs, axis=-1)
            all_preds.extend(preds)
            all_labels.extend(labels_batch)
            all_probs.extend(probs[:, 1])

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs  = np.array(all_probs)

    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()

    acc         = accuracy_score(all_labels, all_preds)
    f1_w        = f1_score(all_labels, all_preds, average="weighted")
    prec_w      = precision_score(all_labels, all_preds, average="weighted")
    rec_w       = recall_score(all_labels, all_preds, average="weighted")
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    auc_roc     = roc_auc_score(all_labels, all_probs)
    auc_pr      = average_precision_score(all_labels, all_probs)
    clf_report  = classification_report(
        all_labels, all_preds,
        target_names=["negatif", "positif"],
        output_dict=True,
        digits=4,
    )

    print("=" * 55)
    print("         MODEL EVALUATION RESULTS")
    print("=" * 55)
    print(f"  Accuracy            : {acc:.4f}")
    print(f"  F1 (weighted)       : {f1_w:.4f}")
    print(f"  Precision (weighted): {prec_w:.4f}")
    print(f"  Recall (weighted)   : {rec_w:.4f}")
    print(f"  Specificity         : {specificity:.4f}")
    print(f"  AUC-ROC             : {auc_roc:.4f}")
    print(f"  AUC-PR              : {auc_pr:.4f}")
    print()
    print(classification_report(all_labels, all_preds, target_names=["negatif", "positif"], digits=4))

    cm    = confusion_matrix(all_labels, all_preds)
    fig1, ax1 = plt.subplots(figsize=(6, 6))
    disp  = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["negatif", "positif"])
    disp.plot(ax=ax1, colorbar=False, cmap="Blues")
    ax1.set_title("Confusion Matrix (Validation Set)", fontsize=13, fontweight="bold")
    fig1.tight_layout()
    cm_path = os.path.join(eval_dir, "indobert_confusion_matrix.png")
    fig1.savefig(cm_path, dpi=150, bbox_inches="tight")
    plt.close(fig1)
    print(f"Saved: {cm_path}")

    fpr, tpr, _ = roc_curve(all_labels, all_probs)
    prec_curve, rec_curve, _ = precision_recall_curve(all_labels, all_probs)

    fig2, (ax2, ax3) = plt.subplots(1, 2, figsize=(12, 5))
    fig2.suptitle("ROC Curve & Precision-Recall Curve - IndoBERT", fontsize=13, fontweight="bold")

    ax2.plot(fpr, tpr, color="#7ea5f8", linewidth=2, label=f"AUC-ROC = {auc_roc:.4f}")
    ax2.plot([0, 1], [0, 1], "k--", linewidth=1)
    ax2.set_xlabel("False Positive Rate")
    ax2.set_ylabel("True Positive Rate")
    ax2.set_title("ROC Curve")
    ax2.legend(loc="lower right")
    ax2.grid(True, alpha=0.3)

    ax3.plot(rec_curve, prec_curve, color="#73e39c", linewidth=2, label=f"AUC-PR = {auc_pr:.4f}")
    ax3.set_xlabel("Recall")
    ax3.set_ylabel("Precision")
    ax3.set_title("Precision-Recall Curve")
    ax3.legend(loc="lower left")
    ax3.grid(True, alpha=0.3)

    fig2.tight_layout()
    roc_pr_path = os.path.join(eval_dir, "indobert_roc_pr_curves.png")
    fig2.savefig(roc_pr_path, dpi=150, bbox_inches="tight")
    plt.close(fig2)
    print(f"Saved: {roc_pr_path}")

    all_metrics = {
        "per_epoch": {
            "train_loss"     : callback.train_loss,
            "train_accuracy" : callback.train_acc,
            "train_precision": callback.train_prec,
            "train_recall"   : callback.train_rec,
            "train_f1"       : callback.train_f1,
            "val_loss"       : callback.val_loss,
            "val_accuracy"   : callback.val_acc,
            "val_precision"  : callback.val_prec,
            "val_recall"     : callback.val_rec,
            "val_f1"         : callback.val_f1,
        },
        "val_set": {
            "accuracy"             : round(acc,         4),
            "f1_weighted"          : round(f1_w,        4),
            "precision_weighted"   : round(prec_w,      4),
            "recall_weighted"      : round(rec_w,       4),
            "specificity"          : round(specificity, 4),
            "auc_roc"              : round(auc_roc,     4),
            "auc_pr"               : round(auc_pr,      4),
            "confusion_matrix"     : cm.tolist(),
            "classification_report": clf_report,
        },
    }
    metrics_path = os.path.join(eval_dir, "indobert_all_metrics.json")
    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(all_metrics, f, indent=2, ensure_ascii=False)
    print(f"Saved: {metrics_path}")
    return all_metrics

device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
val_dataset = ReviewDataset(X_val, y_val, tokenizer)
eval_metrics = evaluate_model(model, val_dataset, device, EVAL_DIR, metrics_callback)

## Phase 3 - Integrated Gradients (IG) Lexicon

In [ ]:
def run_integrated_gradients(reviews, model, tokenizer, device, top_k_per_review: int = 3, min_freq: int = 2) -> set:
    FILTER_OUT = (STOPWORDS | NEGATION_WORDS | set(_string.punctuation) | {"[CLS]", "[SEP]", "[PAD]", "[UNK]"})

    model.eval()
    model.to(device)
    embedding_layer = model.bert.embeddings.word_embeddings

    def forward_func(input_embeds, attention_mask):
        return model(inputs_embeds=input_embeds, attention_mask=attention_mask).logits

    ig            = IntegratedGradients(forward_func)
    token_counter = Counter()
    total         = 0
    print(f"Running Integrated Gradients on {len(reviews)} reviews...")
    for i, review in enumerate(reviews):
        if not review or not review.strip():
            continue
        try:
            enc = tokenizer(review, truncation=True, max_length=128, padding="max_length", return_tensors="pt")
            input_ids    = enc["input_ids"].to(device)
            attn_mask    = enc["attention_mask"].to(device)
            input_embeds = embedding_layer(input_ids).detach().requires_grad_(True)
            baseline     = torch.zeros_like(input_embeds)

            with torch.no_grad():
                pred_class = torch.argmax(model(input_ids=input_ids, attention_mask=attn_mask).logits, dim=-1).item()

            attrs = ig.attribute(input_embeds, baselines=baseline, additional_forward_args=(attn_mask), target=pred_class, n_steps=50)
            scores = attrs.squeeze(0).norm(dim=-1).detach().cpu().numpy()
            toks   = tokenizer.convert_ids_to_tokens(input_ids.squeeze(0).cpu().tolist())

            scored = []
            for tok, sc in zip(toks, scores):
                t = tok.lower().strip()
                if (t in FILTER_OUT or t.startswith("##") or len(t) < 3 or t.isdigit()):
                    continue
                scored.append((t, float(sc)))
            scored.sort(key=lambda x: x[1], reverse=True)
            for t, _ in scored[:top_k_per_review]:
                token_counter[t] += 1
            total += 1
        except Exception:
            continue

        if (i + 1) % 500 == 0:
            print(f"Processed {i + 1}/{len(reviews)}...")
    print(f"Done. {total} reviews processed.")
    candidates = [t for t, c in token_counter.most_common() if c >= min_freq and is_clean_word(t)]

    print()
    print(f"Top IG opinion tokens (min_freq={min_freq}):")
    print("-" * 45)
    for t, c in token_counter.most_common(50):
        if c >= min_freq and is_clean_word(t):
            print(f"  {t:<25} freq={c}")
    return set(candidates)

model.to(device)
model.eval()

IG_LEXICON = run_integrated_gradients(df_train_pool["review_normalized"].tolist(), model, tokenizer, device, top_k_per_review=3, min_freq=2)
print(f"\nIG lexicon contains {len(IG_LEXICON)} tokens.")

lexicon_path = os.path.join(RESULTS_DIR, "indobert_lexicon.json")
with open(lexicon_path, "w", encoding="utf-8") as f:
    json.dump(sorted(IG_LEXICON), f, indent=2, ensure_ascii=False)
print(f"Saved: {lexicon_path}")

## Phase 4 - Parsing

#### Clause Splitting on Conjunctions and Punctuation

In [ ]:
def split_into_clauses(text: str) -> list[str]:
    if not isinstance(text, str) or not text.strip():
        return []
    conj_pattern   = r"\b(?:" + "|".join(map(re.escape, SPLIT_CONJUNCTIONS)) + r")\b"
    processed_text = re.sub(conj_pattern, "|", text, flags=re.IGNORECASE)
    processed_text = re.sub(r"[.,!?;\s*]{2,}|[.,!?;]", "|", processed_text)
    raw_clauses    = processed_text.split("|")
    all_clauses    = []
    for clause in raw_clauses:
        cleaned = re.sub(r"\s+", " ", clause).strip()
        if cleaned:
            all_clauses.append(cleaned)
    return all_clauses

sample_input  = "pengiriman cepat tapi produk jelek,, packing rapi dan admin ramah .. ."
sample_output = split_into_clauses(sample_input)
print(f"Clause splitting example")
print(f"INPUT  : {sample_input}")
print(f"OUTPUT : {sample_output}")

#### POS & Dependency Parsing + Phrase Extraction (Stanza)

In [ ]:
def pos_and_dep_parse(text: str) -> list[dict]:
    if not isinstance(text, str) or not text.strip():
        return []
    doc    = nlp_stanza(text)
    tokens = []
    for sent in doc.sentences:
        for word in sent.words:
            tokens.append({
                "id"     : word.id - 1,
                "word"   : word.lemma.lower() if word.lemma else word.text.lower(),
                "pos"    : word.upos,
                "dep"    : word.deprel,
                "head_id": word.head - 1,
            })
    return tokens

def extract_phrases(clause: str) -> list[dict]:
    tokens = pos_and_dep_parse(clause)
    if not tokens:
        return []

    phrases  = []
    seen     = set()
    n        = len(tokens)
    NOUN_POS = {"NOUN", "PROPN"}
    ADJ_POS  = {"ADJ"}
    ADV_POS  = {"ADV"}
    VERB_POS = {"VERB"}

    def is_neg(tok):
        return tok["word"] in NEGATION_WORDS

    def add(tokens_in_phrase, pattern):
        text = " ".join(t["word"] for t in tokens_in_phrase)
        if text not in seen:
            seen.add(text)
            phrases.append({"phrase": text, "pattern": pattern})

    # Rule 1: NOUN + ADJ
    for i in range(n - 1):
        if tokens[i]["pos"] in NOUN_POS and tokens[i+1]["pos"] in ADJ_POS:
            if not is_neg(tokens[i]) and not is_neg(tokens[i+1]):
                add([tokens[i], tokens[i+1]], "R1_NOUN+ADJ")
    # Rule 2: NOUN + ADV + ADJ
    for i in range(n - 2):
        if (tokens[i]["pos"] in NOUN_POS
                and tokens[i+1]["pos"] in ADV_POS
                and tokens[i+2]["pos"] in ADJ_POS
                and not is_neg(tokens[i+1])):
            add([tokens[i], tokens[i+1], tokens[i+2]], "R2_NOUN+ADV+ADJ")
    # Rule 3: NOUN + NEG + ADJ
    for i in range(n - 2):
        if (tokens[i]["pos"] in NOUN_POS
                and is_neg(tokens[i+1])
                and tokens[i+2]["pos"] in ADJ_POS):
            add([tokens[i], tokens[i+1], tokens[i+2]], "R3_NOUN+NEG+ADJ")
    # Rule 4: NOUN + NEG + ADV + ADJ
    for i in range(n - 3):
        if (tokens[i]["pos"] in NOUN_POS
                and is_neg(tokens[i+1])
                and tokens[i+2]["pos"] in ADV_POS
                and tokens[i+3]["pos"] in ADJ_POS):
            add([tokens[i], tokens[i+1], tokens[i+2], tokens[i+3]], "R4_NOUN+NEG+ADV+ADJ")
    # Rule 5: ADJ + NOUN (inversion)
    for i in range(n - 1):
        if tokens[i]["pos"] in ADJ_POS and tokens[i+1]["pos"] in NOUN_POS:
            if tokens[i+1]["dep"] in {"root", "nsubj"}:
                add([tokens[i+1], tokens[i]], "R5_ADJ+NOUN")
    # Rule 6: NOUN + VERB + ADJ
    for i in range(n - 2):
        if (tokens[i]["pos"] in NOUN_POS
                and tokens[i+1]["pos"] in VERB_POS
                and tokens[i+2]["pos"] in ADJ_POS):
            add([tokens[i], tokens[i+1], tokens[i+2]], "R6_NOUN+VERB+ADJ")
    # Rule 7: NOUN + VERB + NEG + ADJ
    for i in range(n - 3):
        if (tokens[i]["pos"] in NOUN_POS
                and tokens[i+1]["pos"] in VERB_POS
                and is_neg(tokens[i+2])
                and tokens[i+3]["pos"] in ADJ_POS):
            add([tokens[i], tokens[i+1], tokens[i+2], tokens[i+3]], "R7_NOUN+VERB+NEG+ADJ")
    # Rule 8: NOUN + NOUN + ADJ
    for i in range(n - 2):
        if (tokens[i]["pos"] in NOUN_POS
                and tokens[i+1]["pos"] in NOUN_POS
                and tokens[i+2]["pos"] in ADJ_POS):
            add([tokens[i], tokens[i+1], tokens[i+2]], "R8_NOUN+NOUN+ADJ")
    # Rule 9: NOUN + NOUN + NEG + ADJ
    for i in range(n - 3):
        if (tokens[i]["pos"] in NOUN_POS
                and tokens[i+1]["pos"] in NOUN_POS
                and is_neg(tokens[i+2])
                and tokens[i+3]["pos"] in ADJ_POS):
            add([tokens[i], tokens[i+1], tokens[i+2], tokens[i+3]], "R9_NOUN+NOUN+NEG+ADJ")
    # Rule 10: NOUN + ADJ + ADJ
    for i in range(n - 2):
        if (tokens[i]["pos"] in NOUN_POS
                and tokens[i+1]["pos"] in ADJ_POS
                and tokens[i+2]["pos"] in ADJ_POS):
            add([tokens[i], tokens[i+1], tokens[i+2]], "R10_NOUN+ADJ+ADJ")
    # Rule 11: Dependency Conjunction Tree
    for token in tokens:
        if token["dep"] == "conj":
            head_id = token["head_id"]
            if 0 <= head_id < n:
                head = tokens[head_id]
                if head["pos"] in NOUN_POS and token["pos"] in ADJ_POS:
                    add([head, token], "R11_conj+shared_head")
                elif head["pos"] in ADJ_POS and token["pos"] in ADJ_POS:
                    for other in tokens:
                        if other["pos"] in NOUN_POS and other["head_id"] == head_id:
                            add([other, head, token], "R11_conj+shared_head")
                            break

    return phrases

test_clause = "pengiriman sangat cepat"
print("\nPOS + Dependency parse:")
for t in pos_and_dep_parse(test_clause):
    print(f"  {t['word']:15} POS={t['pos']:6} DEP={t['dep']}")
print("\nPhrase extraction:")
for p in extract_phrases(test_clause):
    print(f"  {p}")

## Phase 5 - IG Lexicon Filter

In [ ]:
def filter_by_ig_lexicon(phrase_list: list[dict]) -> list[dict]:
    if not IG_LEXICON:
        print("Warning: IG_LEXICON is empty. Skipping lexicon filter.")
        return phrase_list
    return [p for p in phrase_list if any(w in IG_LEXICON for w in p["phrase"].lower().split())]

## Phase 6 - Phrase Sentiment Prediction

In [ ]:
def predict_sentiment_batch(texts, batch_size=64, max_length=64):
    results = []
    for i in range(0, len(texts), batch_size):
        batch     = texts[i : i + batch_size]
        encodings = tokenizer(batch, truncation=True, padding=True, max_length=max_length, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**encodings)
            probs   = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
        for prob in probs:
            label_id = int(np.argmax(prob))
            results.append({
                "label": ID2LABEL[label_id],
                "score": float(prob[label_id]),
            })
    return results

def predict_phrases_sentiment(phrase_list: list[dict]) -> list[dict]:
    if not phrase_list:
        return []
    texts       = [p["phrase"] for p in phrase_list]
    predictions = predict_sentiment_batch(texts, max_length=32)
    return [
        {
            "phrase"   : pd_item["phrase"],
            "sentiment": pred["label"],
            "score"    : pred["score"],
            "pattern"  : pd_item["pattern"],
        }
        for pd_item, pred in zip(phrase_list, predictions)
        if pred["score"] >= CONFIDENCE_THRESHOLD
    ]

## Phase 7 - Semantic Deduplication (cosine similarity)

In [ ]:
def get_phrase_embeddings(phrases: list[str]) -> np.ndarray:
    embeddings = []
    for i in range(0, len(phrases), 64):
        batch     = phrases[i : i + 64]
        encodings = tokenizer(batch, truncation=True, padding=True, max_length=32, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs        = model(**encodings, output_hidden_states=True)
            cls_embeddings = outputs.hidden_states[-1][:, 0, :].cpu().numpy()
            embeddings.append(cls_embeddings)
    return np.vstack(embeddings)

def semantic_deduplication(phrase_sentiment_list: list[dict], threshold: float = SIMILARITY_THRESHOLD) -> list[dict]:
    if not phrase_sentiment_list:
        return []

    def cluster_phrases(phrases):
        if not phrases:
            return []
        texts          = [p["phrase"] for p in phrases]
        freq           = Counter(texts)
        unique_phrases = list(freq.keys())
        unique_counts  = [freq[p] for p in unique_phrases]

        if len(unique_phrases) <= 1:
            return [{"phrase": unique_phrases[0], "count": unique_counts[0]}] if unique_phrases else []

        embeddings = get_phrase_embeddings(unique_phrases)
        sim_matrix = cosine_similarity(embeddings)
        merged     = [False] * len(unique_phrases)
        result     = []
        for i in range(len(unique_phrases)):
            if merged[i]:
                continue
            cluster_count = unique_counts[i]
            for j in range(i + 1, len(unique_phrases)):
                if not merged[j] and sim_matrix[i][j] >= threshold:
                    cluster_count += unique_counts[j]
                    merged[j]      = True
            result.append({"phrase": unique_phrases[i], "count": cluster_count})
        return result

    print("Running semantic deduplication...")
    pos_clustered = cluster_phrases([p for p in phrase_sentiment_list if p["sentiment"] == "positif"])
    neg_clustered = cluster_phrases([p for p in phrase_sentiment_list if p["sentiment"] == "negatif"])

    return (
        [{**item, "sentiment": "positif"} for item in pos_clustered] +
        [{**item, "sentiment": "negatif"} for item in neg_clustered]
    )

## Phase 8 - Full Pipeline

In [ ]:
def run_full_pipeline(df: pd.DataFrame, top_n: int = TOP_N_SUMMARY, min_count: int = MIN_COUNT):
    all_phrases = []
    total       = len(df)
    start       = time.time()

    print(f"Processing {total} reviews...\n")
    with tqdm(total=total, desc="Phrase Extraction", unit="review") as pbar:
        for i, (_, row) in enumerate(df.iterrows()):
            review  = row["review_normalized"]
            clauses = split_into_clauses(review) or [review]
            for clause in clauses:
                all_phrases.extend(extract_phrases(clause))
            pbar.update(1)
            if i % 10 == 0:
                elapsed = time.time() - start
                rate    = (i + 1) / elapsed if elapsed > 0 else 0
                eta     = (total - i) / rate if rate > 0 else 0
                pbar.set_postfix({
                    "phrases": len(all_phrases),
                    "rate"   : f"{rate:.1f} rev/s",
                    "ETA"    : f"{int(eta//60)}m {int(eta%60):02d}s",
                })
    print()
    elapsed_extract = time.time() - start
    print(f"Extraction complete in {int(elapsed_extract//60)}m {int(elapsed_extract%60):02d}s")
    print(f"Total phrases extracted: {len(all_phrases)}")
    if not all_phrases:
        print("No phrases extracted.")
        return None, None, {"total_extracted": 0, "total_accepted": 0, "total_rejected": 0}

    all_phrases = filter_by_ig_lexicon(all_phrases)
    print(f"Phrases after IG filter: {len(all_phrases)}")

    print()
    print("Predicting phrase sentiment...")
    accepted = []
    with tqdm(total=len(all_phrases), desc="Sentiment Prediction", unit="phrase") as pbar2:
        for i in range(0, len(all_phrases), 64):
            batch  = all_phrases[i : i + 64]
            result = predict_phrases_sentiment(batch)
            accepted.extend(result)
            pbar2.update(len(batch))
            pbar2.set_postfix({
                "accepted": len(accepted),
                "rejected": (i + len(batch)) - len(accepted),
            })

    total_accepted = len(accepted)
    total_rejected = len(all_phrases) - total_accepted

    print()
    print(f"Phrases accepted: {total_accepted}")
    print(f"Phrases rejected: {total_rejected} (confidence < {CONFIDENCE_THRESHOLD})")

    phrase_stats = {
        "total_extracted": len(all_phrases),
        "total_accepted" : total_accepted,
        "total_rejected" : total_rejected,
    }

    deduplicated = semantic_deduplication(accepted)
    pos_raw = sorted([p for p in deduplicated if p["sentiment"] == "positif"], key=lambda x: x["count"], reverse=True)
    neg_raw = sorted([p for p in deduplicated if p["sentiment"] == "negatif"], key=lambda x: x["count"], reverse=True)

    print(f"Unique positive clusters: {len(pos_raw)}")
    print(f"Unique negative clusters: {len(neg_raw)}")

    for p in pos_raw + neg_raw:
        p["pct"] = round(min(p["count"] / total * 100, 100.0), 1)

    pos_phrases = [p for p in pos_raw[:top_n] if p["count"] >= min_count]
    neg_phrases = [p for p in neg_raw[:top_n] if p["count"] >= min_count]

    elapsed_total = time.time() - start
    print()
    print(f"Pipeline complete in {int(elapsed_total//60)}m {int(elapsed_total%60):02d}s")

    return pos_phrases, neg_phrases, phrase_stats

df_sample = df_raw.reset_index(drop=True)
pos_summary, neg_summary, phrase_stats = run_full_pipeline(df_sample, top_n=TOP_N_SUMMARY, min_count=MIN_COUNT)

if pos_summary and neg_summary:
    print("\n" + "=" * 55)
    print("          AUTOMATED REVIEW SUMMARY")
    print("=" * 55)

    print("\nPOSITIVE SUMMARY")
    print("-" * 55)
    for i, p in enumerate(pos_summary, 1):
        bar = "#" * max(1, int(p["pct"] / 2))
        print(f"  {i:2}. {p['phrase']:<25} {bar:<20} {p['pct']}% ({p['count']} occurrences)")

    print("\nNEGATIVE SUMMARY")
    print("-" * 55)
    for i, p in enumerate(neg_summary, 1):
        bar = "#" * max(1, int(p["pct"] / 2))
        print(f"  {i:2}. {p['phrase']:<25} {bar:<20} {p['pct']}% ({p['count']} occurrences)")

    print("\n" + "=" * 55)

## Phase 9 - Save All Outputs

In [ ]:
summary = {
    "phrase_stats": phrase_stats,
    "positive"    : pos_summary or [],
    "negative"    : neg_summary or [],
}
phrase_summary_path = os.path.join(RESULTS_DIR, "indobert_phrase_summary.json")
with open(phrase_summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(f"Saved: {phrase_summary_path}")

print("\nAll outputs:")
for out_dir, label in [(MODEL_DIR, "model_weights"), (EVAL_DIR, "evaluation_metrics"), (RESULTS_DIR, "results")]:
    print(f"\n  [{label}]")
    for fname in sorted(os.listdir(out_dir)):
        fpath = os.path.join(out_dir, fname)
        if os.path.isfile(fpath):
            size_kb = os.path.getsize(fpath) / 1024
            print(f"    {fname:<45} {size_kb:>8.1f} KB")